## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
# Create DataFrame from SQL query
df = spark.sql("SELECT `Order ID`, `Product`, `Quantity Ordered`, `Price Each`, `Order Date`, `Purchase Address` FROM sales_december_2019")

df.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295665|  Macbook Pro Laptop|               1|      1700|12/30/19 00:01|136 Church St, Ne...|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:58|200 Jefferson St,...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|         

Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
# Your code here
dbutils.widgets.text("Product", "USB-C Charging Cable")
product_name = dbutils.widgets.get("Product")
filtered_df = df.filter(df['Product'] == product_name)
filtered_df.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|               2|     11.95|12/13/19 09:29|813 Hickory St, D...|
|  295675|USB-C Charging Cable|               2|     11.95|12/13/19 13:52|594 1st St, San F...|
|  295679|USB-C Charging Cable|               1|     11.95|12/25/19 09:39|902 2nd St, Dalla...|
|  295681|USB-C Charging Cable|               1|     11.95|12/25/19 12:37|79 Elm St, Boston...|
|  295682|USB-C Charging Cable|         

As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
# Your code here
from pyspark.sql.functions import col, sum, round
#getting only nums without Nulls or naming
df_clean = df.filter(col("Quantity Ordered").rlike("^[0-9]+$"))

df_clean = (
    df_clean.withColumn("Quantity Ordered", col("Quantity Ordered").cast("int"))
      .withColumn("Price Each", col("Price Each").cast("double"))
)

#total sales and quantity by product
df_total_sales_product = df_clean.groupBy("Product").agg(
        sum("Quantity Ordered").alias("Total_Quantity_Sold"),
        round(sum(col("Quantity Ordered") * col("Price Each")),2).alias("Total_Sales")
    )
df_total_sales_product.show()

+--------------------+-------------------+-----------+
|             Product|Total_Quantity_Sold|Total_Sales|
+--------------------+-------------------+-----------+
|        20in Monitor|                571|   62804.29|
|     ThinkPad Laptop|                541|  540994.59|
|Bose SoundSport H...|               1825|  182481.75|
|Apple Airpods Hea...|               2079|   311850.0|
|AAA Batteries (4-...|               4240|    12677.6|
|              iPhone|                908|   635600.0|
|        Google Phone|                716|   429600.0|
|            LG Dryer|                 86|    51600.0|
|    27in FHD Monitor|                965|  144740.35|
|34in Ultrawide Mo...|                849|  322611.51|
|AA Batteries (4-p...|               3718|   14277.12|
|    Wired Headphones|               2748|   32948.52|
|USB-C Charging Cable|               3251|   38849.45|
|     Vareebadd Phone|                285|   114000.0|
|27in 4K Gaming Mo...|                861|  335781.39|
|  LG Wash

In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
# Your code here
from pyspark.sql.functions import col, to_timestamp, min, max, count

df_clean = df.filter(col("Quantity Ordered").rlike("^[0-9]+$"))

df_clean = (
    df_clean
      .withColumn("Quantity Ordered", col("Quantity Ordered").cast("int"))
      .withColumn("Price Each", col("Price Each").cast("double"))
      .withColumn("Order Date", to_timestamp("Order Date", "MM/dd/yy HH:mm"))
)

profile_df = df_clean.agg(
    count("*").alias("row_count"),
    min("Order ID").alias("OrderID_min"),
    max("Order ID").alias("OrderID_max"),
    min("Product").alias("Product_min"),
    max("Product").alias("Product_max"),
    min("Quantity Ordered").alias("QuantityOrdered_min"),
    max("Quantity Ordered").alias("QuantityOrdered_max"),
    min("Price Each").alias("PriceEach_min"),
    max("Price Each").alias("PriceEach_max"),
    min("Order Date").alias("OrderDate_min"),
    max("Order Date").alias("OrderDate_max"),
    min("Purchase Address").alias("PurchaseAddress_min"),
    max("Purchase Address").alias("PurchaseAddress_max")
)

profile_df.show()



+---------+-----------+-----------+------------+-----------+-------------------+-------------------+-------------+-------------+-------------------+-------------------+--------------------+--------------------+
|row_count|OrderID_min|OrderID_max| Product_min|Product_max|QuantityOrdered_min|QuantityOrdered_max|PriceEach_min|PriceEach_max|      OrderDate_min|      OrderDate_max| PurchaseAddress_min| PurchaseAddress_max|
+---------+-----------+-----------+------------+-----------+-------------------+-------------------+-------------+-------------+-------------------+-------------------+--------------------+--------------------+
|    24989|     295665|     319670|20in Monitor|     iPhone|                  1|                  7|         2.99|       1700.0|2019-12-01 02:50:00|2020-01-01 05:13:00|1 12th St, San Fr...|999 West St, Los ...|
+---------+-----------+-----------+------------+-----------+-------------------+-------------------+-------------+-------------+-------------------+--------


5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
#your code here
from pyspark.sql.functions import lit

df_clean = df_clean.withColumn("Source", lit("sales_december_2019"))
df_clean.show()

+--------+--------------------+----------------+----------+-------------------+--------------------+-------------------+
|Order ID|             Product|Quantity Ordered|Price Each|         Order Date|    Purchase Address|             Source|
+--------+--------------------+----------------+----------+-------------------+--------------------+-------------------+
|  295665|  Macbook Pro Laptop|               1|    1700.0|2019-12-30 00:01:00|136 Church St, Ne...|sales_december_2019|
|  295666|  LG Washing Machine|               1|     600.0|2019-12-29 07:03:00|562 2nd St, New Y...|sales_december_2019|
|  295667|USB-C Charging Cable|               1|     11.95|2019-12-12 18:21:00|277 Main St, New ...|sales_december_2019|
|  295668|    27in FHD Monitor|               1|    149.99|2019-12-22 15:13:00|410 6th St, San F...|sales_december_2019|
|  295669|USB-C Charging Cable|               1|     11.95|2019-12-18 12:38:00|43 Hill St, Atlan...|sales_december_2019|
|  295670|AA Batteries (4-p...| 

Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
# Your code here for view creation
profile_df.createOrReplaceTempView("sales_december_2019_profile")


In [0]:

%sql
-- Your SQL code here

SELECT row_count, OrderID_min, OrderID_max, product_min, product_max FROM sales_december_2019_profile;

row_count,OrderID_min,OrderID_max,product_min,product_max
24989,295665,319670,20in Monitor,iPhone
